In [1]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np
import time

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os  

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

from requests.exceptions import ConnectTimeout, ReadTimeout, RequestException

In [2]:
date = "2026-02"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

In [3]:
end_use = "naics?get=CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url) 
    
print(r)
    
df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

df.columns = r.json()[0]

df["total_imports"] = df["CON_VAL_MO"].astype(float)

df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])

top_products = grp.agg({"total_imports":"sum","CTY_CODE":"first"})

country_list = list(top_products.sort_values(by = "total_imports", ascending = False).CTY_CODE)[0:31]


['TOTAL FOR ALL COUNTRIES','NAFTA','EUROPEAN UNION']

<Response [200]>


['TOTAL FOR ALL COUNTRIES', 'NAFTA', 'EUROPEAN UNION']

In [4]:
df.tail()

,CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL,time,total_imports
39797,137489598,7940,ZAMBIA,DET,2026-02,137489598.0
39798,1311879,7950,ESWATINI,DET,2026-02,1311879.0
39799,12936682,7960,ZIMBABWE,DET,2026-02,12936682.0
39800,5417103,7970,MALAWI,DET,2026-02,5417103.0
39801,6713622,7990,LESOTHO,DET,2026-02,6713622.0


In [5]:
country_list[0] = ""

In [6]:
country_list.extend(["0003", "0020"])

In [7]:
len(country_list)

33

In [8]:
end_use = "hs?get=CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

surl  = surl + my_key + "&COMM_LVL=HS10" 

for xxx in country_list:
    
    out_file = ".\\data"+ "\\imports-hs10\\" + xxx + "data-" + date + ".parquet"
    
    if xxx == "":
        out_file = ".\\data"+ "\\imports-hs10\\" + "TOTAL" + "data-" + date + ".parquet"
    
    
    if os.path.exists(out_file):
        
        print("Already have downloaded file")
        
        continue
    
    print(f"Downloading {xxx} for {date}")
    
    url = surl + "&time=" + date
    
    if xxx != "":
        url = url + "&CTY_CODE=" + xxx
    
    max_retries = 5
    retry_count = 0
    r = None
    
    while retry_count < max_retries:
        try:
            # Added timeout (30 seconds) and read timeout
            r = requests.get(url, timeout=30)
            
            if r.status_code == 200:
                break
            else:
                print(f"Request failed with status {r.status_code}, waiting 30 seconds...")
                time.sleep(30)
                retry_count += 1
                
        except (ConnectTimeout, ReadTimeout) as e:
            retry_count += 1
            print(f"Connection timeout on attempt {retry_count}/{max_retries}: {type(e).__name__}")
            if retry_count < max_retries:
                wait_time = 30 * (2 ** (retry_count - 1))  # Exponential backoff
                print(f"Waiting {wait_time} seconds before retry...")
                time.sleep(wait_time)
            else:
                print(f"Max retries exceeded for {xxx}, skipping...")
                continue
                
        except RequestException as e:
            print(f"Request failed with error: {e}")
            retry_count += 1
            if retry_count < max_retries:
                time.sleep(30)
    
    if r is None or r.status_code != 200:
        print(f"Failed to download {xxx} after {max_retries} attempts, skipping...")
        continue
    
    print(r)
    
    foo = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

    foo.columns = r.json()[0]

    pq.write_table(pa.Table.from_pandas(foo), out_file)


<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>


In [9]:
# Append dated files to current files and replace them
import glob

data_dir = ".\\data\\imports-hs10"

# Find all current files
current_files = glob.glob(os.path.join(data_dir, "*data-current.parquet"))

for current_file in current_files:
    # Extract the country code from the filename
    filename = os.path.basename(current_file)
    country_code = filename.replace("data-current.parquet", "")
    
    # Find the corresponding dated file
    dated_file = os.path.join(data_dir, f"{country_code}data-{date}.parquet")
    
    if not os.path.exists(dated_file):
        print(f"Dated file not found for {country_code}, skipping...")
        continue
    
    print(f"Processing {country_code}...")
    
    # Read both files
    current_df = pq.read_table(current_file).to_pandas()
    dated_df = pq.read_table(dated_file).to_pandas()
    
    # Append dated data to current
    combined_df = pd.concat([current_df, dated_df], ignore_index=True)
    
    # Write back to current file
    pq.write_table(pa.Table.from_pandas(combined_df), current_file)
    
    print(f"Updated {country_code}: appended {len(dated_df)} rows, total now {len(combined_df)}")

print("Done!")


Processing 0003...
Updated 0003: appended 13555 rows, total now 2250315
Processing 0020...
Updated 0020: appended 11010 rows, total now 1884746
Processing 1220...
Updated 1220: appended 9194 rows, total now 1597795
Processing 2010...
Updated 2010: appended 7378 rows, total now 1314054
Processing 3010...
Updated 3010: appended 2166 rows, total now 395996
Processing 3370...
Updated 3370: appended 919 rows, total now 191529
Processing 3510...
Updated 3510: appended 3055 rows, total now 694256
Processing 4010...
Updated 4010: appended 2991 rows, total now 594465
Processing 4120...
Updated 4120: appended 7228 rows, total now 1338547
Processing 4190...
Updated 4190: appended 1680 rows, total now 333789
Processing 4210...
Updated 4210: appended 4126 rows, total now 825916
Processing 4231...
Updated 4231: appended 3129 rows, total now 674509
Processing 4279...
Updated 4279: appended 6607 rows, total now 1251459
Processing 4280...
Updated 4280: appended 7962 rows, total now 1478239
Processing 4

In [10]:
combined_df.CTY_NAME.unique()

array(['TOTAL FOR ALL COUNTRIES'], dtype=object)

In [11]:
combined_df[combined_df.time == "2026-01"].sort_values(by = "CON_VAL_MO", ascending = False).head(20)

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC,time,COMM_LVL
2723945,TOTAL FOR ALL COUNTRIES,999875,449944,2916391700,"2,2-DICHLOROPHENYLACETIC ACID ETHYL ESTER ETC..",2026-01,HS10
2713033,TOTAL FOR ALL COUNTRIES,99979,17909,8429595080,"MECH SHVL,EXCAVAT,SHV LOAD,EX360 REVOL STUR,USED",2026-01,HS10
2718830,TOTAL FOR ALL COUNTRIES,99977,15756,6202203500,"W/G PADDED SLEEVELESS JACKETS OF WOOL, NOT K N...",2026-01,HS10
2716200,TOTAL FOR ALL COUNTRIES,99961,4973,7228501040,OTH BRS RDS OTH TS CF/FN MX CS UN 18MM OTHR,2026-01,HS10
2711616,TOTAL FOR ALL COUNTRIES,999159,190794,9401612011,"HSHLD SEAT,WDN FRMS,CHAIRS,TEAK,PLNTATION,UPHLST",2026-01,HS10
2722697,TOTAL FOR ALL COUNTRIES,998844,143852,2007914000,ORANGE MARMALADE,2026-01,HS10
2723715,TOTAL FOR ALL COUNTRIES,9988403,1077084,2905120050,PROPAN-2-OL,2026-01,HS10
2723187,TOTAL FOR ALL COUNTRIES,9986394,587364,2523100000,CEMENT CLINKERS,2026-01,HS10
2710436,TOTAL FOR ALL COUNTRIES,9982622,549464,8702906100,"BUS-TYPE VEH, FOR TRANSPORT 10-15 PERSONS, NESOI",2026-01,HS10
2725641,TOTAL FOR ALL COUNTRIES,998226,29336,0706904040,"SALSIFY & SIMILAR EDIBLE ROOTS, FR/CH",2026-01,HS10


In [12]:
foo = pd.read_parquet(".\\data\\imports-hs10\\0003data-current.parquet")

foo[foo.time == "2026-01"].sort_values(by = "CON_VAL_MO", ascending = False).head(20)

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC,time,COMM_LVL,CTY_CODE
2226269,EUROPEAN UNION,9998966,4160,3912390000,"CELLULOSE ETHERS,NESOI",2026-01,HS10,0003
2232647,EUROPEAN UNION,99988,14999,2005707525,"OLIVES NOT GREEN, NOT CANNED, IN SALINE, NESOI",2026-01,HS10,0003
2229501,EUROPEAN UNION,99974,15002,6209205050,"BABIES' OT GRMNTS & CLOTHING ACCESS OF COTTON,...",2026-01,HS10,0003
2230660,EUROPEAN UNION,998891,149719,8536304000,"MOTOR OVERLOAD PROTECTORS, FOR VOLTAGE LT=1000V",2026-01,HS10,0003
2234850,EUROPEAN UNION,99886,14988,6117809540,"CLOTHING ACCESSORIES OF MANMADE FIBERS,NESOI, ...",2026-01,HS10,0003
2233523,EUROPEAN UNION,99880,12882,9011208000,"MICROSCOPES, EXC WITH MEANS TO PHOTOGRAPH IMAGE",2026-01,HS10,0003
2226221,EUROPEAN UNION,9987230,1463888,3907995050,"POLYESTERS,NESOI",2026-01,HS10,0003
2224739,EUROPEAN UNION,9985,1497,0305420060,"HERRINGS, SMOKED, NESOI",2026-01,HS10,0003
2226822,EUROPEAN UNION,99835,14975,2918232000,ODORIF OR FLAVOR CMPDS OF SALTS ETC OF SALICY ...,2026-01,HS10,0003
2235848,EUROPEAN UNION,998331,44423,7216910010,"ANGL,SHP,SECT IOS NA,CLD-FRMD/FNSHD FLT-RL,DRI...",2026-01,HS10,0003


In [13]:
current_files

['.\\data\\imports-hs10\\0003data-current.parquet',
 '.\\data\\imports-hs10\\0020data-current.parquet',
 '.\\data\\imports-hs10\\1220data-current.parquet',
 '.\\data\\imports-hs10\\2010data-current.parquet',
 '.\\data\\imports-hs10\\3010data-current.parquet',
 '.\\data\\imports-hs10\\3370data-current.parquet',
 '.\\data\\imports-hs10\\3510data-current.parquet',
 '.\\data\\imports-hs10\\4010data-current.parquet',
 '.\\data\\imports-hs10\\4120data-current.parquet',
 '.\\data\\imports-hs10\\4190data-current.parquet',
 '.\\data\\imports-hs10\\4210data-current.parquet',
 '.\\data\\imports-hs10\\4231data-current.parquet',
 '.\\data\\imports-hs10\\4279data-current.parquet',
 '.\\data\\imports-hs10\\4280data-current.parquet',
 '.\\data\\imports-hs10\\4330data-current.parquet',
 '.\\data\\imports-hs10\\4419data-current.parquet',
 '.\\data\\imports-hs10\\4621data-current.parquet',
 '.\\data\\imports-hs10\\4700data-current.parquet',
 '.\\data\\imports-hs10\\4759data-current.parquet',
 '.\\data\\i

In [14]:
# Combine all current data files into one big dataset
all_data = []

for current_file in current_files:
    # Extract the country code from the filename
    filename = os.path.basename(current_file)
    country_code = filename.replace("data-current.parquet", "")
    
    print(f"Reading {country_code}...")
    
    # Read the file
    df_country = pq.read_table(current_file).to_pandas()
    
    # Add country code column if not already present
    if 'CTY_CODE' not in df_country.columns:
        df_country['CTY_CODE'] = country_code
    
    all_data.append(df_country)

# Combine all countries into one dataframe
bigdf = pd.concat(all_data, ignore_index=True)

print(f"\nCombined dataset: {len(bigdf):,} rows, {len(all_data)} countries")

# Save to parquet
output_file = ".\\data\\imports-hs10\\ALL-data-current.parquet"
pq.write_table(pa.Table.from_pandas(bigdf), output_file)
print(f"Saved to: {output_file}")

Reading 0003...
Reading 0020...
Reading 1220...
Reading 2010...
Reading 3010...
Reading 3370...
Reading 3510...
Reading 4010...
Reading 4120...
Reading 4190...
Reading 4210...
Reading 4231...
Reading 4279...
Reading 4280...
Reading 4330...
Reading 4419...
Reading 4621...
Reading 4700...
Reading 4759...
Reading 5081...
Reading 5170...
Reading 5330...
Reading 5490...
Reading 5520...
Reading 5570...
Reading 5590...
Reading 5600...
Reading 5700...
Reading 5800...
Reading 5830...
Reading 5880...
Reading 6021...
Reading ALL-...
Reading TOTAL...

Combined dataset: 65,263,022 rows, 34 countries
Saved to: .\data\imports-hs10\ALL-data-current.parquet


In [15]:
bigdf.CTY_NAME.unique()

array(['EUROPEAN UNION', 'USMCA (NAFTA)', 'CANADA', 'MEXICO', 'COLOMBIA',
       'CHILE', 'BRAZIL', 'SWEDEN', 'UNITED KINGDOM', 'IRELAND',
       'NETHERLANDS', 'BELGIUM', 'FRANCE', 'GERMANY', 'AUSTRIA',
       'SWITZERLAND', 'RUSSIA', 'SPAIN', 'ITALY', 'ISRAEL',
       'SAUDI ARABIA', 'INDIA', 'THAILAND', 'VIETNAM', 'MALAYSIA',
       'SINGAPORE', 'INDONESIA', 'CHINA', 'KOREA, SOUTH', 'TAIWAN',
       'JAPAN', 'AUSTRALIA', 'TOTAL FOR ALL COUNTRIES'], dtype=object)

In [16]:
foo = bigdf[bigdf.CTY_NAME == "EUROPEAN UNION"].copy()

foo[foo.time == "2026-01"].sort_values(by = "CON_VAL_MO", ascending = False).head(20)

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC,time,COMM_LVL,CTY_CODE
32204656,EUROPEAN UNION,9998966,4160,3912390000,"CELLULOSE ETHERS,NESOI",2026-01,HS10,0003
2226269,EUROPEAN UNION,9998966,4160,3912390000,"CELLULOSE ETHERS,NESOI",2026-01,HS10,0003
32211034,EUROPEAN UNION,99988,14999,2005707525,"OLIVES NOT GREEN, NOT CANNED, IN SALINE, NESOI",2026-01,HS10,0003
2232647,EUROPEAN UNION,99988,14999,2005707525,"OLIVES NOT GREEN, NOT CANNED, IN SALINE, NESOI",2026-01,HS10,0003
2229501,EUROPEAN UNION,99974,15002,6209205050,"BABIES' OT GRMNTS & CLOTHING ACCESS OF COTTON,...",2026-01,HS10,0003
32207888,EUROPEAN UNION,99974,15002,6209205050,"BABIES' OT GRMNTS & CLOTHING ACCESS OF COTTON,...",2026-01,HS10,0003
32209047,EUROPEAN UNION,998891,149719,8536304000,"MOTOR OVERLOAD PROTECTORS, FOR VOLTAGE LT=1000V",2026-01,HS10,0003
2230660,EUROPEAN UNION,998891,149719,8536304000,"MOTOR OVERLOAD PROTECTORS, FOR VOLTAGE LT=1000V",2026-01,HS10,0003
2234850,EUROPEAN UNION,99886,14988,6117809540,"CLOTHING ACCESSORIES OF MANMADE FIBERS,NESOI, ...",2026-01,HS10,0003
32213237,EUROPEAN UNION,99886,14988,6117809540,"CLOTHING ACCESSORIES OF MANMADE FIBERS,NESOI, ...",2026-01,HS10,0003
